栅格数据处理（扩展学习）    
1. 栅格数据重采样    
2. 栅格数据重投影（第四章节内容复习）  

1. 重采样

In [7]:
import rasterio as rio
from rasterio.transform import Affine
from rasterio.warp import reproject, Resampling


In [8]:
path_rsimg = 'data/rsimg/s2_20251206.tif'
path_rsimg_resam = 'data/rsimg/s2_20251206_resam.tif'


In [9]:
# --- 设置目标分辨率（可自定义X/Y方向不同）---
target_x_res = 21.0     # 目标X方向分辨率（米）
target_y_res = 21.0     # 目标Y方向分辨率（米） 


In [10]:
## 读取原始栅格
with rio.open(path_rsimg) as src:
    data = src.read(1)       # 读取第一波段
    src_transform = src.transform  # 原始仿射变换矩阵
    src_crs = src.crs        # 原始坐标系
    src_width = src.width    # 原始宽度（像素）
    src_height = src.height  # 原始高度（像素）

print(src_transform)  # x方向分辨率，x方向偏移，左上角点x坐标；y方向...
# data.shape
print(src_crs)  


| 20.00, 0.00, 251920.00|
| 0.00,-20.00, 2770520.00|
| 0.00, 0.00, 1.00|
EPSG:32648


In [21]:
# --- 1. 获取原始分辨率（X/Y方向）---
x_res = src_transform.a   # X方向分辨率（例如10米）
y_res = -src_transform.e  # Y方向分辨率（取正值，例如10米）

# --- 2. 计算重采样栅格参数 ---
# 2.1 缩放比例（宽度/高度）
x_scale = x_res / target_x_res  # X方向缩放比例
y_scale = y_res / target_y_res  # Y方向缩放比例

# 2.2 重采样栅格尺寸（整数像素）
dst_width = int(src_width * x_scale)
dst_height = int(src_height * y_scale)

# 2.3 重采样栅格仿射变换矩阵（调整分辨率，保持左上角原点不变）
dst_transform = Affine(
    a = target_x_res,     # X方向分辨率
    b = src_transform.b,  # X方向旋转（通常为0）
    c = src_transform.c,  # 左上角X坐标
    d = src_transform.d,  # Y方向旋转（通常为0）
    e = -target_y_res,    # Y方向分辨率（负值表示北向）
    f = src_transform.f   # 左上角Y坐标
    )


In [ ]:
# --- 创建目标栅格并执行重采样 ---
with rio.open(
    path_rsimg_resam,
    "w",
    driver="GTiff",
    height=dst_height,
    width=dst_width,
    count=1, 
    dtype=data.dtype,
    crs=src_crs,
    transform=dst_transform,
    ) as dst:

    reproject(
        source=data,  # 原数据
        src_transform=src_transform,  # 原数据仿射变换矩阵    
        src_crs=src_crs,              # 原数据坐标系    
        destination=rio.band(dst, 1),     # 目标数据（rio.band():波段写入）   
        dst_transform=dst_transform,      # 目标数据仿射变换矩阵   
        dst_crs=src_crs,                  # 目标数据坐标系   
        resampling=Resampling.bilinear,   # 可替换为其他方法   
    )


2. 重投影

In [23]:
import rasterio as rio
from rasterio.warp import reproject, Resampling, calculate_default_transform  


In [ ]:
from pyproj import CRS
path_rsimg = 'data/rsimg/s2_20251206.tif'
path_rsimg_reproj = 'data/rsimg/s2_20251206_reproj2wgs84.tif'
# target_crs = 'EPSG:4326'  ## 目标坐标系（WGS84）,epsg格式rasterio支持不足
target_crs = CRS.from_epsg(4326)  ## 目标坐标系（WGS84）
target_crs = target_crs.to_wkt()   # WKT 格式


In [ ]:
with rio.open(path_rsimg) as src:
    src_data = src.read()        # 读取第一波段
    src_crs = src.crs            # 原始坐标系
    src_count = src.count        # 波段数
    src_nodata = src.nodata      # 原始NoData值
    src_dtype = src.dtypes[0]    # 数据类型（假设所有波段一致）
    src_bounds = src.bounds      # 原始地理范围（left, bottom, right, top）
    src_transform = src.transform  
    

In [ ]:
# 1. 计算重投影后影像参数（转换矩阵、宽度、高度）
dst_transform, dst_width, dst_height = calculate_default_transform(
    src_crs = src_crs,          # 原坐标系
    dst_crs = target_crs,       # 目标坐标系（WGS84）
    width = src.width,          # 原始宽度（像素）
    height = src.height,        # 原始高度（像素）
    left = src_bounds.left,
    bottom = src_bounds.bottom,
    right = src_bounds.right,
    top = src_bounds.top 
    )

dst_transform  

Affine(0.0001853149319713278, 0.0, 102.54145549031928,
       0.0, -0.0001853149319713278, 25.03432635505885)

In [27]:
# 创建目标栅格文件
with rio.open(
    fp=path_rsimg_reproj,
    mode="w",
    driver="GTiff",
    height=dst_height,
    width=dst_width,
    count=src_count,  # 单波段
    dtype=src_data.dtype,
    crs=target_crs,
    transform=dst_transform) as dst:
    # 执行重投影（无需循环，直接处理单波段）
    for i in range(1, src_count + 1):
        reproject(
            source=src_data[i - 1],  # 输入数据
            src_transform=src_transform,
            src_crs=src_crs,
            src_nodata=src_nodata,
            destination=rio.band(dst, i),  # 目标波段
            dst_transform=dst_transform,
            dst_crs=target_crs,
            resampling=Resampling.bilinear,  # 重采样方法
        )
    # 关闭目标文件
    dst.close()
